In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

In [2]:
from google.colab import files
uploaded = files.upload()

Saving hand_landmarks_data.csv to hand_landmarks_data.csv


In [22]:
df = pd.read_csv('hand_landmarks_data.csv')

In [23]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Features and target
X = df.drop(['label'], axis=1).values

# Convert labels to integers
le = LabelEncoder()
y = le.fit_transform(df['label'])

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)


In [24]:
# Compute class weights for imbalanced classes
from sklearn.utils import class_weight
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train), y=y_train
)
class_weights = dict(enumerate(class_weights_array))


In [25]:
# Build the model
model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),

    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(len(np.unique(y)), activation='softmax')
])

In [26]:
# Compile the model with a lower learning rate
from tensorflow.keras.optimizers import Adam
optimizer = Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])



In [27]:
# Add early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


In [28]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=2
)

Epoch 1/100
257/257 - 6s - 23ms/step - accuracy: 0.4823 - loss: 1.6012 - val_accuracy: 0.7442 - val_loss: 1.0073
Epoch 2/100
257/257 - 2s - 10ms/step - accuracy: 0.7420 - loss: 0.6716 - val_accuracy: 0.8452 - val_loss: 0.4263
Epoch 3/100
257/257 - 4s - 14ms/step - accuracy: 0.8093 - loss: 0.4948 - val_accuracy: 0.8788 - val_loss: 0.2997
Epoch 4/100
257/257 - 4s - 15ms/step - accuracy: 0.8339 - loss: 0.4132 - val_accuracy: 0.9002 - val_loss: 0.2625
Epoch 5/100
257/257 - 2s - 10ms/step - accuracy: 0.8507 - loss: 0.3715 - val_accuracy: 0.9080 - val_loss: 0.2299
Epoch 6/100
257/257 - 3s - 10ms/step - accuracy: 0.8659 - loss: 0.3413 - val_accuracy: 0.9192 - val_loss: 0.2101
Epoch 7/100
257/257 - 3s - 14ms/step - accuracy: 0.8770 - loss: 0.3131 - val_accuracy: 0.9287 - val_loss: 0.1917
Epoch 8/100
257/257 - 4s - 16ms/step - accuracy: 0.8840 - loss: 0.2963 - val_accuracy: 0.9362 - val_loss: 0.1810
Epoch 9/100
257/257 - 3s - 10ms/step - accuracy: 0.8961 - loss: 0.2785 - val_accuracy: 0.9326 - 

In [29]:
# Evaluate the model
y_pred = np.argmax(model.predict(X_test), axis=1)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}")

161/161 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.97      0.97       301
           1       1.00      1.00      1.00       259
           2       1.00      0.99      1.00       189
           3       0.98      0.99      0.99       327
           4       0.98      0.98      0.98       287
           5       0.94      0.97      0.95       217
           6       1.00      0.99      1.00       318
           7       0.98      0.96      0.97       253
           8       0.98      0.96      0.97       330
           9       0.98      0.98      0.98       288
          10       0.99      0.98      0.99       299
          11       1.00      0.99      0.99       292
          12       0.94      0.98      0.96       296
          13       0.99      1.00      1.00       314
          14       0.99      0.97      0.98       291
          15       1.00      0.99      1.00       331
          16   

In [31]:
model.save("enhanced_model.keras")